In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s  - дата и время события
#%(levelname)s - уровень: INFO в нашем случае
#%(message)s  - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig() - основная функция для настройки logging'
    #уровень INFO - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a' - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests** - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas** - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1 - Дневные свечи (candles)

**Цель:** получить историю дневных цен АЛРОСЫ за период 2018-01-01 - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи АЛРОСА с MOEX (первые 500 строк)
response_ALRS = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_ALRS = response_ALRS.json()

#извлекаем данные из json
columns_ALRS = data_ALRS['candles']['columns']
rows_ALRS = data_ALRS['candles']['data']

df_ALRS = pd.DataFrame(rows_ALRS, columns=columns_ALRS)
df_ALRS['ticker'] = 'ALRS'

print(f'Count of rows: {len(df_ALRS)}')
#считаем все строки - мало ли их много, а выводит только 500
df_ALRS


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,75.21,75.81,75.96,75.15,2.953372e+08,3908500,2018-01-03 00:00:00,2018-01-03 23:59:59,ALRS
1,75.79,77.00,77.54,75.78,9.858285e+08,12833400,2018-01-04 00:00:00,2018-01-04 23:59:59,ALRS
2,77.42,76.90,77.42,76.21,4.509697e+08,5883800,2018-01-05 00:00:00,2018-01-05 23:59:59,ALRS
3,77.30,76.79,77.90,76.50,1.018036e+09,13198500,2018-01-09 00:00:00,2018-01-09 23:59:59,ALRS
4,76.99,76.70,77.40,76.47,8.426125e+08,10944800,2018-01-10 00:00:00,2018-01-10 23:59:59,ALRS
...,...,...,...,...,...,...,...,...,...
495,80.50,80.66,80.93,80.04,7.546751e+08,9371130,2019-12-16 00:00:00,2019-12-16 23:59:59,ALRS
496,80.85,81.74,82.13,80.20,1.463088e+09,17989970,2019-12-17 00:00:00,2019-12-17 23:59:59,ALRS
497,81.60,81.84,82.10,81.38,8.502841e+08,10399870,2019-12-18 00:00:00,2019-12-18 23:59:59,ALRS
498,81.90,84.52,84.52,81.60,2.437928e+09,29185860,2019-12-19 00:00:00,2019-12-19 23:59:59,ALRS


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open  - цена акции АЛРОСЫ в начале торгового дня (рублей за акцию)
-  close  - цена в конце торгового дня
-  high  - максимальная цена за день
-  low  - минимальная цена за день
-  value  - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume  - количество акций сменивших владельца за день
-  begin  - дата и время начала торгового дня
-  end  - дата и время конца торгового дня
-  ticker  - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_ALRS = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_ALRS = response_ALRS.json()
print(f"Count of rows after 500 rows: {len(data_ALRS['candles']['data'])}")
#строк много - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_ALRSnew = data_ALRS['candles']['columns']
rows_ALRSnew = data_ALRS['candles']['data']

df_ALRSnew = pd.DataFrame(rows_ALRSnew, columns=columns_ALRSnew)
df_ALRSnew['ticker'] = 'ALRS'

print(f'Count of rows after 500 rows: {len(df_ALRSnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_ALRS2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_ALRS2 = response_ALRS2.json()
columns_ALRSnew2 = data_ALRS2['candles']['columns']
rows_ALRSnew2 = data_ALRS2['candles']['data']

df_ALRSnew2 = pd.DataFrame(rows_ALRSnew2, columns=columns_ALRSnew2)
df_ALRSnew2['ticker'] = 'ALRS'

print(f'Count of rows after 1000 rows: {len(df_ALRSnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_ALRS3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_ALRS3 = response_ALRS3.json()
columns_ALRSnew3 = data_ALRS3['candles']['columns']
rows_ALRSnew3 = data_ALRS3['candles']['data']

df_ALRSnew3 = pd.DataFrame(rows_ALRSnew3, columns=columns_ALRSnew3)
df_ALRSnew3['ticker'] = 'ALRS'

print(f'Count of rows after 1500 rows: {len(df_ALRSnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_ALRS4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_ALRS4 = response_ALRS4.json()
columns_ALRSnew4 = data_ALRS4['candles']['columns']
rows_ALRSnew4 = data_ALRS4['candles']['data']

df_ALRSnew4 = pd.DataFrame(rows_ALRSnew4, columns=columns_ALRSnew4)
df_ALRSnew4['ticker'] = 'ALRS'

print(f'Count of rows after 2000 rows: {len(df_ALRSnew4)}')

Count of rows after 2000 rows: 73


In [ ]:
#проверяем есть ли данные ниже 2073 строки
logging.info('Запрос 1: проверяем есть ли данные после 2073 строки (start=2073)')
response_ALRS5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2073
      })

data_ALRS5 = response_ALRS5.json()
columns_ALRSnew5 = data_ALRS5['candles']['columns']
rows_ALRSnew5 = data_ALRS5['candles']['data']

df_ALRSnew5 = pd.DataFrame(rows_ALRSnew5, columns=columns_ALRSnew5)
df_ALRSnew5['ticker'] = 'ALRS'

print(f'Count of rows after 2073 rows: {len(df_ALRSnew5)}')

Count of rows after 2073 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_ALRScandles = pd.concat([df_ALRS, df_ALRSnew, df_ALRSnew2, df_ALRSnew3, df_ALRSnew4], ignore_index=True)
print(f'Total rows: {len(df_ALRScandles)}')

logging.info(f'Запрос 1: итого после склейки всех частей пяти датафреймов - {len(df_ALRScandles)} строк')
df_ALRScandles

Total rows: 2073


,open,close,high,low,value,volume,begin,end,ticker
0,75.21,75.81,75.96,75.15,2.953372e+08,3908500,2018-01-03 00:00:00,2018-01-03 23:59:59,ALRS
1,75.79,77.00,77.54,75.78,9.858285e+08,12833400,2018-01-04 00:00:00,2018-01-04 23:59:59,ALRS
2,77.42,76.90,77.42,76.21,4.509697e+08,5883800,2018-01-05 00:00:00,2018-01-05 23:59:59,ALRS
3,77.30,76.79,77.90,76.50,1.018036e+09,13198500,2018-01-09 00:00:00,2018-01-09 23:59:59,ALRS
4,76.99,76.70,77.40,76.47,8.426125e+08,10944800,2018-01-10 00:00:00,2018-01-10 23:59:59,ALRS
...,...,...,...,...,...,...,...,...,...
2068,42.48,43.00,43.29,42.29,5.718020e+08,13315990,2025-12-26 00:00:00,2025-12-26 23:59:58,ALRS
2069,43.07,42.96,43.29,42.92,3.699991e+07,858430,2025-12-27 00:00:00,2025-12-27 23:59:57,ALRS
2070,43.00,42.95,43.06,42.62,5.146090e+07,1200980,2025-12-28 00:00:00,2025-12-28 23:59:57,ALRS
2071,42.80,41.56,43.53,41.00,1.308671e+09,31076000,2025-12-29 00:00:00,2025-12-29 23:59:59,ALRS


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2073 строки данных нет. Значит итоговый датафрейм  df_ALRScandles  содержит полную историю дневных котировок АЛРОСЫ за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2073
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True  - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **2073 строк** - это все торговые дни АЛРОСЫ за период с 01.01.2018 по 31.12.2025.



## Запрос 2 - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям АЛРОСА: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги АЛРОСА
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_ALRS_infa = session.get(
    'https://iss.moex.com/iss/securities/ALRS.json')

data_ALRS_infa = response_ALRS_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_ALRS_infa = data_ALRS_infa['description']['columns']
rows_ALRS_infa = data_ALRS_infa['description']['data']

df_ALRS_infa = pd.DataFrame(rows_ALRS_infa, columns=columns_ALRS_infa)
df_ALRS_infa['ticker'] = 'ALRS'

print(f'Count of rows: {len(df_ALRS_infa)}')
df_ALRS_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,ALRS,string,1,0,NaN,ALRS
1,ISSUENAME,Наименование ценной бумаги,акции обыкновенные,string,2,0,NaN,ALRS
2,NAME,Полное наименование,АЛРОСА ПАО ао,string,3,0,NaN,ALRS
3,SHORTNAME,Краткое наименование,АЛРОСА ао,string,4,0,NaN,ALRS
4,ISIN,ISIN код,RU0007252813,string,5,0,NaN,ALRS
5,REGNUMBER,Номер государственной регистрации,1-03-40046-N,string,6,0,NaN,ALRS
6,ISSUESIZE,Объем выпуска,7364965630,number,7,0,0.0,ALRS
7,FACEVALUE,Номинальная стоимость,0.5,number,8,0,2.0,ALRS
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,ALRS
9,ISSUEDATE,Дата начала торгов,2011-11-29,date,10,0,NaN,ALRS


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_ALRS_infa_best = df_ALRS_infa.set_index('name')['value'].to_frame().T
df_ALRS_infa_best['ticker'] = 'ALRS'
df_ALRS_infa_best = df_ALRS_infa_best.reset_index()

print(f'Полное название: {df_ALRS_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_ALRS_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_ALRS_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_ALRS_infa_best["ISIN"].values[0]}, листинг={df_ALRS_infa_best["LISTLEVEL"].values[0]}')

df_ALRS_infa_best

Полное название: АЛРОСА ПАО ао
ISIN: RU0007252813
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,ALRS,акции обыкновенные,АЛРОСА ПАО ао,АЛРОСА ао,RU0007252813,1-03-40046-N,7364965630,0.5,SUR,...,1,1,1,2011-08-25,Акция обыкновенная,stock_shares,common_share,Акции,1670,ALRS


**Результат:**
   
Полное название: АЛРОСА ПАО ао
ISIN: RU0007252813
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «АЛРОСА ПАО ао»** - официальное юридическое название компании.
- **ISIN = RU0007252813** - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1** - первый уровень листинга на Мосбирже. Это значит, что АЛРОСА прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3 - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат АЛРОСЫ за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов ALRS')
#запрос 3: получаем историю дивидендных выплат АЛРОСА
#дивиденды влияют на котировки примерно на размер дивиденда
ALRS_diva = session.get(
    'https://iss.moex.com/iss/securities/ALRS/dividends.json')

ALRS_diva_infa = ALRS_diva.json()

#извлекаем данные из json
columns_ALRS_diva = ALRS_diva_infa['dividends']['columns']
rows_ALRS_diva = ALRS_diva_infa['dividends']['data']

df_ALRS_diva = pd.DataFrame(rows_ALRS_diva, columns=columns_ALRS_diva)
df_ALRS_diva['ticker'] = 'ALRS'

print(f'Count of rows: {len(df_ALRS_diva)}')

logging.info(f'Запрос 3: получено {len(df_ALRS_diva)} дивидендных выплат')
df_ALRS_diva

Count of rows: 14


,secid,isin,registryclosedate,value,currencyid,ticker
0,ALRS,RU0007252813,2014-07-18,1.47,RUB,ALRS
1,ALRS,RU0007252813,2015-07-15,1.47,RUB,ALRS
2,ALRS,RU0007252813,2016-07-19,2.09,RUB,ALRS
3,ALRS,RU0007252813,2017-07-20,8.93,RUB,ALRS
4,ALRS,RU0007252813,2018-07-14,5.24,RUB,ALRS
5,ALRS,RU0007252813,2018-10-15,5.93,RUB,ALRS
6,ALRS,RU0007252813,2019-07-15,4.11,RUB,ALRS
7,ALRS,RU0007252813,2019-10-14,3.84,RUB,ALRS
8,ALRS,RU0007252813,2020-07-13,2.63,RUB,ALRS
9,ALRS,RU0007252813,2021-07-04,9.54,RUB,ALRS


**Результат:**  Count of rows: 14

API вернул 14 выплат дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate** - дата закрытия реестра.
- **value** - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_ALRS_diva['registryclosedate'] = pd.to_datetime(df_ALRS_diva['registryclosedate'])
df_ALRS_diva_srok = df_ALRS_diva[
    (df_ALRS_diva['registryclosedate'] >= '2018-01-01') &
    (df_ALRS_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_ALRS_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_ALRS_infa_best["ISIN"].values[0]}, листинг={df_ALRS_infa_best["LISTLEVEL"].values[0]}')
df_ALRS_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 10


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,4,ALRS,RU0007252813,2018-07-14,5.24,RUB,ALRS
1,5,ALRS,RU0007252813,2018-10-15,5.93,RUB,ALRS
2,6,ALRS,RU0007252813,2019-07-15,4.11,RUB,ALRS
3,7,ALRS,RU0007252813,2019-10-14,3.84,RUB,ALRS
4,8,ALRS,RU0007252813,2020-07-13,2.63,RUB,ALRS
5,9,ALRS,RU0007252813,2021-07-04,9.54,RUB,ALRS
6,10,ALRS,RU0007252813,2021-10-19,8.79,RUB,ALRS
7,11,ALRS,RU0007252813,2023-10-18,3.77,RUB,ALRS
8,12,ALRS,RU0007252813,2024-05-31,2.02,RUB,ALRS
9,13,ALRS,RU0007252813,2024-10-19,2.49,RUB,ALRS


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 10


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4 - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по АЛРОСЕ - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные ALRS')
#запрос 4: получаем текущие торговые данные АЛРОСА
response_ALRS_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ALRS.json')

data_ALRS_markdata = response_ALRS_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_ALRS_markdata = data_ALRS_markdata['marketdata']['columns']
rows_ALRS_markdata = data_ALRS_markdata['marketdata']['data']

df_ALRS_markdata = pd.DataFrame(rows_ALRS_markdata, columns=columns_ALRS_markdata)
df_ALRS_markdata['ticker'] = 'ALRS'

print(f'Count of rows: {len(df_ALRS_markdata)}')

logging.info(f'Запрос 4: получено {len(df_ALRS_markdata)} строк market data')
df_ALRS_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,ALRS,TQBR,38.44,None,38.45,None,0.01,393398,293321,38.53,...,2026-03-19 23:15:38,38.48,2570,2.831829e+11,23:14:54,None,731508406,2,-662846906.7,ALRS


**Результат:**  Count of rows: 1

Одна строка - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_ALRS_markdata.columns]

df_ALRS_markdata_best = df_ALRS_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена ALRS: {df_ALRS_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_ALRS_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_ALRS_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена ALRS={df_ALRS_markdata_best["LAST"].values[0]}')

df_ALRS_markdata_best

Последняя цена ALRS: 38.45
Изменение к пред. закрытию: -0.23
Капитализация: 283182928473.5


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,ALRS,38.45,-0.23,38.44,38.45,38.53,38.71,38.11,38.42,28634,19041350,731508406,2.831829e+11,23:00:38


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена ALRS: 38.45
Изменение к пред. закрытию: -0.23
Капитализация: 283403877442.4
   

Что означают колонки которые мы оставили:
- **LAST** - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE** - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER** - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом - чем он уже, тем ликвиднее бумага
- **WAPRICE** - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES** - количество сделок за торговый день. У АЛРОСЫ это десятки тысяч - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY** - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION** - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности АЛРОСЫ на текущий момент.

## Запрос 5 - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит АЛРОСА.

Индексные бумаги - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы ALRS')
#запрос 5: получаем список биржевых индексов в которые входит АЛРОСА
response_ALRS_idishka = session.get(
    'https://iss.moex.com/iss/securities/ALRS/indices.json')

data_ALRS_idishka = response_ALRS_idishka.json()

#извлекаем данные из json
columns_ALRS_idishka = data_ALRS_idishka['indices']['columns']
rows_ALRS_idishka = data_ALRS_idishka['indices']['data']

df_ALRS_idishka = pd.DataFrame(rows_ALRS_idishka, columns=columns_ALRS_idishka)
df_ALRS_idishka['ticker'] = 'ALRS'

print(f'Count of rows: {len(df_ALRS_idishka)}')

logging.info(f'Запрос 5: ALRS входит в {len(df_ALRS_idishka)} индексов')

df_ALRS_idishka

Count of rows: 23


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2013-03-18,2026-03-18,1665.38,0.00,-0.02,ALRS
1,ESGI,Индекс МосБиржи Рейтинги УР,2026-02-16,2026-03-18,988.87,-0.96,-9.62,ALRS
2,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,ALRS
3,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,ALRS
4,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,ALRS
5,MCXSM,Индекс МосБиржи SMID,2014-01-06,2014-12-30,1446.12,-0.86,-12.54,ALRS
6,MESG,Индекс МосБиржи-RAEX ESG,2025-01-17,2026-03-18,984.75,-0.27,-2.69,ALRS
7,MOEXBC,Индекс голубых фишек,2015-09-16,2024-09-19,19057.56,-0.03,-5.43,ALRS
8,MRRT,Индекс MRRT,2020-09-21,2026-03-18,2335.80,0.05,1.15,ALRS
9,MRSV,Индекс MRSV,2020-09-21,2026-03-18,2173.86,-0.22,-4.80,ALRS


**Результат:**  Count of rows: 23


Важные колонки:
- **SECID** - код индекса
- **FROM / TILL** - период с которого по который бумага входит в индекс
- **CURRENTVALUE** - текущее значение самого индекса

In [ ]:
#проверяем входит ли ALRS в IMOEX (главный российский фондовый индекс)
if len(df_ALRS_idishka) > 0:
    imoex_check = df_ALRS_idishka[df_ALRS_idishka['SECID'] == 'IMOEX']
    print(f'ALRS входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('ALRS не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили - все данные по ALRS собраны')


ALRS входит в IMOEX: True


**Результат:**
   
ALRS входит в IMOEX: True
   

АЛРОСА входит в IMOEX - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** АЛРОСА входит в 23 индексов включая главный IMOEX. В EDA это будет важным признаком при анализе силы реакции на новости.



